# Lesson 14 Lab — Autotune Search and Experiment Budget

**Puzzle:** When configuration sets, cache keys, and search cost change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates configuration sets, cache keys, and search cost and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

triton.autotune benchmarks eligible configurations when a key changes and caches the winner. This turns meta-parameter selection into measured search, but the first call now includes multiple compilations and executions.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["configuration sets, cache keys, and search cost"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

An autotuned kernel that mutates its output may need a reset hook because every candidate executes during search.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 14
LESSON_TITLE = 'Autotune Search and Experiment Budget'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260827
}


## 5. Freeze the experiment

**Experiment:** Search three affine BLOCK/warp configurations and separate first host cost from cached warm latency.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 570.6414231099188,
  "secondary": 0.033263999968767166,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "configs": 3,
    "warm_samples_ms": [
      0.03734400123357773,
      0.03731200098991394,
      0.033376000821590424,
      0.0331839993596077,
      0.034015998244285583,
      0.035360001027584076,
      0.038816001266241074,
      0.032896000891923904,
      0.03510399907827377,
      0.03359999880194664,
      0.03136000037193298,
      0.032127998769283295,
      0.03222399950027466,
      0.033344000577926636,
      0.03254399821162224,
      0.0315839983522892,
      0.030527999624609947,
      0.03283200040459633,
      0.03177599981427193,
      0.03340800106525421
    ]
  }
}
Three configurations were eligible. The first host call cost 570.64 ms, while the cached warm median was 0.0333 ms.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| First autotune call | 570.6414 ms |
| Cached warm median | 0.0333 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

Three configurations were eligible. The first host call cost 570.64 ms, while the cached warm median was 0.0333 ms.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Spend autotune budget only on dimensions that materially change the optimum and include search cost in deployment planning.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 14,
  "title": "Autotune Search and Experiment Budget",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260827
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 570.6414231099188,
    "secondary": 0.033263999968767166,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "configs": 3,
      "warm_samples_ms": [
        0.03734400123357773,
        0.03731200098991394,
        0.033376000821590424,
        0.0331839993596077,
        0.034015998244285583,
        0.035360001027584076,
        0.038816001266241074,
        0.032896000891923904,
        0.03510399907827377,
        0.03359999880194664,
        0.03136000037193298,
        0.032127998769283295,
        0.03222399950027466,


## 10. Make the bounded decision

> Spend autotune budget only on dimensions that materially change the optimum and include search cost in deployment planning.

**Failure analysis:** An autotuned kernel that mutates its output may need a reset hook because every candidate executes during search.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
